# Real data and real models

Works on `<repo>/data/` and `<repo>/prompts/`. Always run in a **fresh kernel** (never in the
kernel of `test_pipeline.ipynb`). Order: check exports -> prepare data -> check prompts -> smoke test.

In [ ]:
import os
import sys

from dotenv import load_dotenv

load_dotenv()  # e.g. Hugging Face token / cache location
if "config" in sys.modules:
    raise RuntimeError("config is already imported in this kernel: restart the kernel and run from the top")
os.environ.pop("KUKI_TEST", None)  # make sure test mode is off
import config

os.chdir(config.ROOT)  # so that %run finds the scripts
sorted(p.name for p in config.RAW_DIR.iterdir())

## 1. Structural check of the exports
Confirm field names and Label Studio user ids before trusting `prep_00`. The users must match
`config.ANNOTATOR_IDS` (RU A=8, B=4, C=10; TR A=11, B=7, C=6).

In [ ]:
import json
from collections import Counter


def inspect_export(path):
    tasks = json.loads(path.read_text(encoding="utf-8"))
    annotations = [a for t in tasks for a in t.get("annotations", [])]
    print(f"{path.name}: {len(tasks)} tasks, {len(annotations)} annotations")
    print("  task data keys:", sorted(tasks[0]["data"].keys()))
    print("  Label Studio users:", dict(Counter(str(a["completed_by"]) for a in annotations)))
    print("  controls:", dict(Counter(r["from_name"] for a in annotations for r in a["result"])))


for filename in config.EXPORTS.values():
    inspect_export(config.RAW_DIR / filename)

## 2. Data preparation and splits
Compare with the dataset description: articles with 1/2/3 annotators RU 139/42/26, TR 151/44/35;
RU 23 never annotated; 19 self-duplicates in total.

In [ ]:
%run prep_00_parse_exports.py

In [ ]:
%run prep_01_make_splits.py

## 3. Prompt check
What the model sees for one paragraph-with-context call in the native prompt language.

In [ ]:
import llm_io

article = llm_io.load_split("dev")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=1):
    print(f"--- {message['role']} ---\n{message['content'][:1500]}\n")

## 4. Smoke test with a real model
Loads the model with transformers and runs a few constrained calls on one dev article.
Check: valid JSON, token counts, seconds per call (x number of calls = runtime of a stage).

In [ ]:
import time

from tqdm.auto import tqdm

MODEL = "olmo3-7b"

start = time.time()
backend = llm_io.load_model(MODEL)
print(f"model loaded in {time.time() - start:.0f} s, dtype {backend['model'].dtype}")

jobs = llm_io.make_jobs("smoke", [article], MODEL, ["L1", "L2", "L3"], ["doc"], ["en"])
for job in tqdm(jobs, desc="smoke test"):
    record = llm_io.call_llm(backend, job)
    tqdm.write(f"{job['layer']} {record['parsed']} | tokens in/out: {record['tokens_in']}/{record['tokens_out']} "
               f"| {record['seconds']} s | error: {record['error']}")